In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import datetime
import os
import tpqoa as tpqoa 

# **Creating Iterative Base Class**
class IterativeBase():
    
    def __init__(self,symbol,start,end,amount, use_spread=True):
        self.symbol = symbol
        self.position = 0
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.use_spread = use_spread
        self.trades = 0
        self.get_data()
        
    def get_data(self):
        api = tpqoa.tpqoa("D:\\Dissertation Mohit\\Test Environment\\oanda.cfg")
        mid_EURUSD = api.get_history(instrument=self.symbol, start=self.start, end=self.end, granularity='M15', price='M')
        bid_EURUSD = api.get_history(instrument=self.symbol, start=self.start, end=self.end, granularity='M15', price='B')
        ask_EURUSD = api.get_history(instrument=self.symbol, start=self.start, end=self.end, granularity='M15', price='A')
        
        raw = pd.DataFrame({
            'price': mid_EURUSD['c'],
            'spread': ask_EURUSD['c'] - bid_EURUSD['c'],
        })
        
        raw['returns'] = np.log(raw.price/raw.price.shift(1))
        self.data = raw.dropna()
        
    
    def plot_data(self,cols=None):
        if cols is None:
            cols = ['price']
        plt.figure(figsize=(12,6))
        for col in cols:
            plt.plot(self.data[col], label=col)
        plt.title(f'{self.symbol} Price')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()
        
        
    def get_values(self,bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar],5)
        spread = round(self.data.spread.iloc[bar],5)
        return date, price, spread
    
    def get_current_balance(self,bar):
        date,price,spread = self.get_values(bar)
        print(f"Date: {date} | Current Balance: {self.current_balance:.2f}")
        
    def buy_instrument(self,bar,units = None, amount=None):
        date,price,spread = self.get_values(bar)
        if self.use_spread:
            price += spread/2 #ask price
        if amount is not None:
            units = int(amount/price)
        self.current_balance -= units * price
        self.units += units
        self.trades += 1
        
        print(f"Date: {date} | Bought {units} units at {price:.5f} | Current Balance: {self.current_balance:.2f}")
        
    def print_current_position_value(self,bar):
        date,price,spread = self.get_values(bar)
        cpv = self.units * price
        print(f"Date: {date} | Current Position Value: {cpv:.2f}")
        
    def net_asset_value(self,bar):
        date,price,spread = self.get_values(bar)
        nav = self.current_balance + (self.units * price)
        print(f"Date: {date} | Net Asset Value: {nav:.2f}")
        
    def sell_instrument(self,bar,units = None, amount=None):
        date,price,spread = self.get_values(bar)
        if self.use_spread:
            price -= spread/2 #bid price
        if amount is not None:
            units = int(amount/price)
        self.current_balance += units * price
        self.units -= units
        self.trades += 1
        
        print(f"Date: {date} | Sold {units} units at {price:.5f} | Current Balance: {self.current_balance:.2f}")

        
    def close_pos(self,bar):
        date, price, spread = self.get_values(bar)
        print(75*'-')
        print(f"Date: {date} | Closing Position of {self.units} units at {price:.5f}")
        self.current_balance += self.units * price
        self.current_balance -= (abs(self.units)*spread/2) * self.use_spread
        print(f'{date} | closing position of {self.units} for {price}')
        self.units = 0
        self.trades += 1
       
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100 
        self.get_current_balance(bar)
        print(f"Net Performance: {perf:.2f}%")
        print(f'Number of trades executed: {self.trades}')
        print(75*'-')
                
    
bc  = IterativeBase('EUR_USD', '2026-01-01', '2026-03-07', 100000, use_spread=True)

bc.buy_instrument(0,units = 1000)
bc.units
bc.get_current_balance(0)
bc.buy_instrument(1, amount = 5000)
bc.net_asset_value(1)
bc.print_current_position_value(1)
bc.get_current_balance(1)
bc.units
bc.get_values(0)
bc.get_values(100)
bc.close_pos(100)
bc.plot_data()
bc.data
# **Child Class for specific strategies**
class IterativeBacktest(IterativeBase):
    
    #helper method   
    def go_long(self,bar,units = None, amount=None):
        if self.position == -1:
            self.buy_instrument(bar, units = -self.units)
        if units:
            self.buy_instrument(bar, units = units)
        elif amount:
            if amount == 'all':
                amount = self.current_balance
            self.buy_instrument(bar, amount = amount)
        
    def go_short(self,bar,units = None, amount=None):
        if self.position == 1:
            self.sell_instrument(bar, units = self.units)
        if units:
            self.sell_instrument(bar, units = units)
        elif amount:
            if amount == 'all':
                amount = self.current_balance
            self.sell_instrument(bar, amount = amount)
            
    def test_logreg_strategy(self,lags = 5, model_path = None, test_days = 7):
            """
            Test Logistic Regression strategy with iterative backtesting
    
            Parameters:
            -----------
            lags : int
                Number of lagged returns to use as features
            model_path : str
                Path to saved model (optional). If None, trains new model
            test_days : int
                Number of days to use for testing (default 7)
            """
            from sklearn.linear_model import LogisticRegression
            from sklearn.multiclass import OneVsRestClassifier
            
            # Prepare features
            cols = []
            for lag in range(1, lags + 1):
                col = f'lag_{lag}'
                self.data[col] = self.data['returns'].shift(lag)
                cols.append(col)
            
            self.data['direction'] = np.sign(self.data['returns'])
            self.data.dropna(inplace=True)
            
            # Normalize features
            mean = self.data[cols].mean()
            std = self.data[cols].std().replace(0, 1)
            self.data.loc[:, cols] = (self.data[cols] - mean) / std
            
            # Split data: last test_days for testing, rest for training
            test_start_date = self.data.index[-1] - pd.Timedelta(days=test_days)
            train_data = self.data[self.data.index < test_start_date].copy()
            test_data = self.data[self.data.index >= test_start_date].copy()
            
            print(f"\nTraining samples: {len(train_data)}")
            print(f"Testing samples: {len(test_data)}")
            print(f"Training period: {train_data.index[0]} to {train_data.index[-1]}")
            print(f"Testing period: {test_data.index[0]} to {test_data.index[-1]}")
    
            # Train model if not provided
            if model_path is None:
                print("\nTraining Logistic Regression model...")
                lm = OneVsRestClassifier(LogisticRegression(C=1e6, max_iter=100000))
                lm.fit(train_data[cols], train_data['direction'])
                print("Model training complete!")
            else:
                import joblib
                lm = joblib.load(model_path)
                print(f"Model loaded from {model_path}")
            
            # Make predictions on test data
            test_data['pred'] = lm.predict(test_data[cols])
            
            print("\n" + "="*75)
            print("STARTING LOGISTIC REGRESSION STRATEGY BACKTEST")
            print(f"Testing on last {test_days} days")
            print("="*75)
            
            # Reset state for backtest
            self.current_balance = self.initial_balance
            self.units = 0
            self.trades = 0
            self.position = 0
    
            # Iterate through test data
            for i, (idx, row) in enumerate(test_data.iterrows()):
                # Get actual bar index in self.data
                bar = self.data.index.get_loc(idx)
                
                # Get prediction
                prediction = row['pred']
                
                # Execute trades based on prediction
                if prediction == 1:  # Go long
                    if self.position == 0:
                        self.go_long(bar, amount='all')
                        self.position = 1
                    elif self.position == -1:
                        self.go_long(bar, amount='all')
                        self.position = 1
                        
                elif prediction == -1:  # Go short
                    if self.position == 0:
                        self.go_short(bar, amount='all')
                        self.position = -1
                    elif self.position == 1:
                        self.go_short(bar, amount='all')
                        self.position = -1
    
            # Close final position
            if self.units != 0:
                bar = len(self.data) - 1
                self.close_pos(bar)
            
            # Calculate hit ratio
            hits = np.sign(test_data['pred'] * test_data['direction'])
            hit_ratio = hits.value_counts().get(1.0, 0) / len(hits) * 100
            
            print(f"\nHit Ratio: {hit_ratio:.2f}%")
            print(f"Total predictions: {len(test_data)}")
            print(f"Correct predictions: {hits.value_counts().get(1.0, 0)}")
            
            return lm, test_data
        
    # ...existing code...

    def test_random_forest_with_tuning(
        self,
        lags=5,
        test_days=7,
        cv=5,
        tracking_uri="http://localhost:5000",
        experiment_name="Random Forest Forex Strategy",
        random_state=42,
        param_grid=None,
        top_n_cv_runs=10
    ):
        """
        Train/tune RandomForest on train set and test on last `test_days`.
        Logs parent + nested CV runs and best estimator to MLflow.
        """
        import mlflow
        import mlflow.sklearn
        from mlflow.models import infer_signature
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import GridSearchCV
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

        # 1) Feature engineering on copy (avoid mutating self.data)
        df = self.data.copy()
        cols = []
        for lag in range(1, lags + 1):
            col = f"lag_{lag}"
            df[col] = df["returns"].shift(lag)
            cols.append(col)

        df["direction"] = np.sign(df["returns"])
        df = df.dropna().copy()
        df = df[df["direction"] != 0].copy()   # keep {-1, +1}

        # 2) Time split: last N days as test
        test_start_date = df.index[-1] - pd.Timedelta(days=test_days)
        train_data = df[df.index < test_start_date].copy()
        test_data = df[df.index >= test_start_date].copy()

        if train_data.empty or test_data.empty:
            raise ValueError("Train/Test split is empty. Increase history window or reduce test_days.")

        # 3) Normalize with TRAIN stats only
        mean = train_data[cols].mean()
        std = train_data[cols].std().replace(0, 1)
        train_data.loc[:, cols] = (train_data[cols] - mean) / std
        test_data.loc[:, cols] = (test_data[cols] - mean) / std

        X_train, y_train = train_data[cols], train_data["direction"]
        X_test, y_test = test_data[cols], test_data["direction"]

        # 4) Hyperparameter grid
        if param_grid is None:
            param_grid = {
                "n_estimators": [100, 200, 300],
                "max_depth": [None, 5, 10],
                "min_samples_split": [2, 5],
                "min_samples_leaf": [1, 2],
                "max_features": ["sqrt", "log2"]
            }

        mlflow.set_tracking_uri(tracking_uri)
        mlflow.set_experiment(experiment_name)

        with mlflow.start_run(run_name=f"rf_gridsearch_lags_{lags}_testdays_{test_days}") as parent_run:
            # Parent run params
            mlflow.log_param("model", "RandomForestClassifier")
            mlflow.log_param("lags", lags)
            mlflow.log_param("test_days", test_days)
            mlflow.log_param("cv_folds", cv)
            mlflow.log_param("train_rows", len(train_data))
            mlflow.log_param("test_rows", len(test_data))
            mlflow.log_param("train_start", str(train_data.index.min()))
            mlflow.log_param("train_end", str(train_data.index.max()))
            mlflow.log_param("test_start", str(test_data.index.min()))
            mlflow.log_param("test_end", str(test_data.index.max()))

            # Grid search
            base_model = RandomForestClassifier(random_state=random_state, n_jobs=-1)
            grid = GridSearchCV(
                estimator=base_model,
                param_grid=param_grid,
                scoring="accuracy",
                cv=cv,
                n_jobs=-1,
                verbose=1,
                refit=True,
                return_train_score=True
            )
            grid.fit(X_train, y_train)

            # Log top-N CV candidates as nested runs
            cv_results = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
            for _, row in cv_results.head(top_n_cv_runs).iterrows():
                with mlflow.start_run(run_name=f"cv_rank_{int(row['rank_test_score'])}", nested=True):
                    params = row["params"]
                    for k, v in params.items():
                        mlflow.log_param(k, v)
                    mlflow.log_metric("cv_mean_accuracy", float(row["mean_test_score"]))
                    mlflow.log_metric("cv_std_accuracy", float(row["std_test_score"]))
                    mlflow.log_metric("cv_mean_train_score", float(row.get("mean_train_score", np.nan)))

            # Best estimator + test metrics
            best_model = grid.best_estimator_
            y_pred = best_model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
            rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
            f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

            mlflow.log_params(grid.best_params_)
            mlflow.log_metric("best_cv_accuracy", float(grid.best_score_))
            mlflow.log_metric("test_accuracy", float(acc))
            mlflow.log_metric("test_precision_macro", float(prec))
            mlflow.log_metric("test_recall_macro", float(rec))
            mlflow.log_metric("test_f1_macro", float(f1))

            signature = infer_signature(X_train, best_model.predict(X_train))
            mlflow.sklearn.log_model(
                sk_model=best_model,
                artifact_path="best_model",
                signature=signature,
                input_example=X_train.head(5)
            )

        print(f"Best Params: {grid.best_params_}")
        print(f"Best CV Accuracy: {grid.best_score_:.4f}")
        print(f"Test Accuracy: {acc:.4f}")

        test_data = test_data.copy()
        test_data["pred_rf"] = y_pred
        return best_model, grid, test_data, cv_results

    # ...existing code...




In [ ]:

# Create backtest instance
ibt = IterativeBacktest('EUR_USD', '2026-01-01', '2026-03-07', 100000, use_spread=True)
# Run the logistic regression strategy (last 7 days for testing)
model, results = ibt.test_logreg_strategy(lags=5, test_days=7)
ibt = IterativeBacktest("EUR_USD", "2026-01-01", "2026-03-07", 100000, use_spread=True)
best_model, grid, rf_results, cv_results = ibt.test_random_forest_with_tuning(
    lags=5,
    test_days=7,
    cv=5,
    tracking_uri="http://localhost:5000",
    experiment_name="Random Forest Forex Strategy"
)
